# 1.Environment Prepare

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
import pandas
import numpy
import matplotlib

In [ ]:
pip install sdv

In [ ]:
pip install sdmetrics

In [ ]:
pip install -U scikit-learn

# 2.Generate traffic information

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

N = 3000  # “真实人口”样本量（演示用）
# ---- 2.1 个人信息表（profiles_real.csv） ----
profiles = pd.DataFrame({
    "pid": np.arange(N),
    "age": rng.integers(18, 70, size=N),
    "sex": rng.choice(["M", "F"], size=N, p=[0.52, 0.48]),
    "occupation": rng.choice(["student","worker","retiree","unemployed"], size=N, p=[0.25,0.55,0.12,0.08]),
    "income_level": rng.choice(["low","mid","high"], size=N, p=[0.35,0.5,0.15]),
    "home_zone": rng.integers(0, 30, size=N),       # 0.30 的居住区
    "car_ownership": rng.choice([0,1], size=N, p=[0.65,0.35]),
    "season_ticket": rng.choice([0,1], size=N, p=[0.55,0.45])
})

# ---- 2.2 将一天离散为 96 个 15min 时间片，生成“活动/方式”序列（bins_real.csv）----
# 活动与方式词表（可按需扩展）
activities = ["home","work","school","shopping","leisure","other"]
modes = ["stay","walk","bike","bus","metro","car","taxi"]

def synthesize_one_day(profile):
    """根据人口画像，合成 96 个时间片的 (activity, mode)"""
    T = 96
    act = np.full(T, "home", dtype=object)
    mode = np.full(T, "stay", dtype=object)

    age = profile["age"]; occ = profile["occupation"]
    car = profile["car_ownership"]; pass_card = profile["season_ticket"]

    # 粗略的日程分布（演示用）
    if occ == "student":
        leave = rng.integers(28, 36)   # 07:00-09:00
        back  = rng.integers(60, 68)   # 15:00-17:00
        school_bins = (leave+3, back-3)
        act[school_bins[0]:school_bins[1]] = "school"
    elif occ == "worker":
        leave = rng.integers(24, 36)   # 06:00-09:00
        back  = rng.integers(64, 76)   # 16:00-19:00
        work_bins = (leave+4, back-4)
        act[work_bins[0]:work_bins[1]] = "work"
    elif occ == "retiree":
        # 白天一些 leisure/shopping
        l1 = rng.integers(40, 56)      # 10:00-14:00
        l2 = l1 + rng.integers(2, 6)
        act[l1:l2] = rng.choice(["shopping","leisure"])
    else:
        # unemployed：随机外出短时段
        l1 = rng.integers(44, 60)      # 11:00-15:00
        l2 = l1 + rng.integers(2, 4)
        act[l1:l2] = rng.choice(["shopping","other"])

    # 出行方式：在“活动切换前后”填充出行段
    for t in range(1, T):
        if act[t] != act[t-1]:
            # 在切换前后插入 1-2 个出行片段
            s = max(0, t-2); e = min(T, t+1)
            chosen = None
            # 简单的方式选择：有车高概率开车；有月票提高公交概率
            if car == 1 and rng.random() < 0.6:
                chosen = "car"
            else:
                if pass_card == 1 and rng.random() < 0.6:
                    chosen = rng.choice(["bus","metro"])
                else:
                    chosen = rng.choice(["walk","bike","bus","metro","taxi"])
            mode[s:e] = chosen
            # 出行时段的 activity 仍保持“切换后的活动”或“插值”，这里不再细化

    return act, mode

rows = []
for i in range(N):
    a, m = synthesize_one_day(profiles.iloc[i])
    for t in range(96):
        rows.append({"pid": i, "t": t, "activity": a[t], "mode": m[t]})

bins = pd.DataFrame(rows)

profiles.to_csv("profiles_real.csv", index=False)
bins.to_csv("bins_real.csv", index=False)
print(profiles.head(), "\n---\n", bins.head())


# Generate traffic information

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata
import pandas as pd

profiles_real = pd.read_csv("profiles_real.csv")

metadata_instance = SingleTableMetadata()
metadata_instance.detect_from_dataframe(data=profiles_real)
meta = metadata_instance
# 你也可以人工指定每个字段类型/类别，获得更稳的结果
synth = TVAESynthesizer(meta, epochs=200)  # 演示设少一点；实际可 200+
synth.fit(profiles_real)

# 采样 5000 个“虚拟人”
profiles_syn = synth.sample(5000)
profiles_syn.to_csv("profiles_syn.csv", index=False)
profiles_syn.head(100)

In [ ]:
# 运行诊断程序。该报告会执行一些基本的有效性检查，例如：
# 确保主键始终唯一
# 验证合成数据是否在正确的范围内（或类别选项内）
# ETC
from sdmetrics.reports.single_table import DiagnosticReport

diagnostic = DiagnosticReport()
diagnostic.generate(profiles_real, profiles_syn, meta.to_dict())

In [ ]:
loss_df = synth.get_loss_values()
loss_df

In [ ]:
# from matplotlib import pyplot as plt
# _df_10['Loss'].plot(kind='line', figsize=(8, 4), title='Loss')
# plt.gca().spines[['top', 'right']].set_visible(False)

# 准备词表与编号（为 Transformer 做映射）

In [ ]:
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

profiles_real = pd.read_csv("profiles_real.csv")
bins_real = pd.read_csv("bins_real.csv")

# 类别字典
cat_maps = {}
def build_map(vals):
    uniq = sorted(pd.Series(vals).astype(str).unique().tolist())
    return {v:i for i,v in enumerate(uniq)}

cat_maps["sex"]        = build_map(profiles_real["sex"])
cat_maps["occupation"] = build_map(profiles_real["occupation"])
cat_maps["income"]     = build_map(profiles_real["income_level"])
cat_maps["activity"]   = build_map(bins_real["activity"])
cat_maps["mode"]       = build_map(bins_real["mode"])

# 连续变量分箱（age -> 6 桶；home_zone -> 30 桶示例）
age_bins = KBinsDiscretizer(n_bins=6, encode="ordinal", strategy="quantile")
profiles_real["age_bin"] = age_bins.fit_transform(profiles_real[["age"]]).astype(int)

zone_bins = KBinsDiscretizer(n_bins=30, encode="ordinal", strategy="uniform")
profiles_real["home_zone_bin"] = zone_bins.fit_transform(profiles_real[["home_zone"]]).astype(int)

# 将类别映射为 id
profiles_real["sex_id"]        = profiles_real["sex"].map(cat_maps["sex"])
profiles_real["occ_id"]        = profiles_real["occupation"].map(cat_maps["occupation"])
profiles_real["inc_id"]        = profiles_real["income_level"].map(cat_maps["income"])

bins_real["act_id"]  = bins_real["activity"].map(cat_maps["activity"])
bins_real["mode_id"] = bins_real["mode"].map(cat_maps["mode"])

profiles_real[["pid","age_bin","sex_id","occ_id","inc_id","home_zone_bin"]].head(), \
bins_real[["pid","t","act_id","mode_id"]].head()


# 构建数据集与 DataLoader

In [ ]:
import torch, numpy as np
from torch.utils.data import Dataset, DataLoader

class DayDataset(Dataset):
    def __init__(self, profiles_df, bins_df, vocabs):
        self.profiles = profiles_df
        self.bins = bins_df
        self.samples = []
        g = self.bins.groupby("pid")
        for pid, df in g:
            df = df.sort_values("t")
            if len(df) != 96:  # 保证完整一天
                continue
            prof = self.profiles[self.profiles.pid==pid]
            if prof.empty:
                continue
            p = prof.iloc[0]
            prof_tokens = np.array([
                p.age_bin, p.sex_id, p.occ_id, p.inc_id, p.home_zone_bin,
                int(p.car_ownership), int(p.season_ticket)
            ], dtype=np.int64)
            act = df["act_id"].values.astype(np.int64)
            mode = df["mode_id"].values.astype(np.int64)
            self.samples.append((prof_tokens, act, mode))

        self.vocabs = vocabs
        self.BOS_ACT  = vocabs["activity_size"]  # BOS 设为词表末尾
        self.BOS_MODE = vocabs["mode_size"]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        prof, act, mode = self.samples[idx]
        # teacher forcing：输入为前一时刻，预测当前
        x_act = np.concatenate([[self.BOS_ACT],  act[:-1]])
        x_mode= np.concatenate([[self.BOS_MODE], mode[:-1]])
        y_act = act
        y_mode= mode
        return (
            torch.tensor(prof, dtype=torch.long),
            torch.tensor(x_act, dtype=torch.long),
            torch.tensor(x_mode, dtype=torch.long),
            torch.tensor(y_act, dtype=torch.long),
            torch.tensor(y_mode, dtype=torch.long),
        )

# 词表尺寸（+1 给 BOS）
vocabs = {
    "activity_size": len(cat_maps["activity"]),
    "mode_size": len(cat_maps["mode"]),
    "sex_size": len(cat_maps["sex"]),
    "occ_size": len(cat_maps["occupation"]),
    "inc_size": len(cat_maps["income"]),
    "age_bins": 6,
    "zone_bins": 30
}
vocabs["activity_plus_bos"] = vocabs["activity_size"] + 1
vocabs["mode_plus_bos"]     = vocabs["mode_size"] + 1

dataset = DayDataset(
    profiles_real,
    bins_real,
    vocabs
)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)
len(dataset)


# 条件 Transformer 模型

In [ ]:
import torch.nn as nn

class CondTransformer(nn.Module):
    def __init__(self, vocabs, d=256, nhead=8, nlayers=6, prof_fields=7):
        super().__init__()
        self.d = d
        self.act_emb  = nn.Embedding(vocabs["activity_plus_bos"], d)
        self.mode_emb = nn.Embedding(vocabs["mode_plus_bos"], d)
        self.pos_emb  = nn.Embedding(96, d)

        # 各 profile 字段嵌入后求和
        self.age_emb   = nn.Embedding(vocabs["age_bins"], d)
        self.sex_emb   = nn.Embedding(vocabs["sex_size"], d)
        self.occ_emb   = nn.Embedding(vocabs["occ_size"], d)
        self.inc_emb   = nn.Embedding(vocabs["inc_size"], d)
        self.zone_emb  = nn.Embedding(vocabs["zone_bins"], d)
        self.car_emb   = nn.Embedding(2, d)
        self.pass_emb  = nn.Embedding(2, d)

        enc_layer = nn.TransformerEncoderLayer(d_model=d, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)

        self.dec_act  = nn.Linear(d, vocabs["activity_size"])
        self.dec_mode = nn.Linear(d, vocabs["mode_size"])

    def profile_embed(self, prof_tokens):
        # prof_tokens: [B, 7] -> age, sex, occ, inc, zone, car, pass
        age, sex, occ, inc, zone, car, pas = \
            prof_tokens[:,0], prof_tokens[:,1], prof_tokens[:,2], prof_tokens[:,3], prof_tokens[:,4], prof_tokens[:,5], prof_tokens[:,6]
        h = (
            self.age_emb(age) + self.sex_emb(sex) + self.occ_emb(occ) +
            self.inc_emb(inc) + self.zone_emb(zone) + self.car_emb(car) + self.pass_emb(pas)
        )
        return h  # [B, d]

    def forward(self, prof_tokens, x_act, x_mode):
        B, T = x_act.shape
        pos = torch.arange(T, device=x_act.device).unsqueeze(0).expand(B, T)

        h = self.act_emb(x_act) + self.mode_emb(x_mode) + self.pos_emb(pos)
        p = self.profile_embed(prof_tokens).unsqueeze(1).expand(B, T, self.d)
        h = h + p

        h = self.encoder(h)  # [B, T, d]
        return self.dec_act(h), self.dec_mode(h)


# 训练（含简单一致性正则，可按需加）

In [ ]:
import torch, torch.nn.functional as F
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CondTransformer(vocabs).to(device)
opt = AdamW(model.parameters(), lr=2e-4)

def step(train=True):
    model.train(train)
    total = 0.0
    for prof, x_act, x_mode, y_act, y_mode in train_loader:
        prof, x_act, x_mode, y_act, y_mode = \
            prof.to(device), x_act.to(device), x_mode.to(device), y_act.to(device), y_mode.to(device)

        logits_act, logits_mode = model(prof, x_act, x_mode)
        loss_act  = F.cross_entropy(logits_act.reshape(-1, logits_act.size(-1)),  y_act.reshape(-1))
        loss_mode = F.cross_entropy(logits_mode.reshape(-1, logits_mode.size(-1)),y_mode.reshape(-1))

        # 简单一致性正则：鼓励“活动保持” → 减少频繁抖动（可选）
        # reg = ((logits_act[:,:-1].argmax(-1) == logits_act[:,1:].argmax(-1)).float().mean())
        # loss = loss_act + loss_mode - 0.01*reg  # 演示：别让它主导

        loss = loss_act + loss_mode

        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    return total / len(train_loader)

for epoch in range(50):  # 演示训练 8 轮；实际可 30+
    L = step(train=True)
    print(f"epoch {epoch+1}: train loss = {L:.4f}")


# 基于“虚拟人画像”进行生成（推断）

In [ ]:
import pandas as pd
import torch

# 从 Stage A 采样的虚拟人口
profiles_syn = pd.read_csv("profiles_syn.csv")

# 用与真实数据相同的分箱/映射处理“虚拟人”
profiles_syn["age_bin"]       = age_bins.transform(profiles_syn[["age"]]).astype(int)
profiles_syn["home_zone_bin"] = zone_bins.transform(profiles_syn[["home_zone"]]).astype(int)
profiles_syn["sex_id"]        = profiles_syn["sex"].map(cat_maps["sex"])
profiles_syn["occ_id"]        = profiles_syn["occupation"].map(cat_maps["occupation"])
profiles_syn["inc_id"]        = profiles_syn["income_level"].map(cat_maps["income"])

def generate_day_for_profile(p):
    prof_tokens = torch.tensor([[
        int(p.age_bin), int(p.sex_id), int(p.occ_id), int(p.inc_id),
        int(p.home_zone_bin), int(p.car_ownership), int(p.season_ticket)
    ]], dtype=torch.long, device=device)

    x_act  = torch.full((1,1), fill_value=vocabs["activity_size"], dtype=torch.long, device=device) # BOS
    x_mode = torch.full((1,1), fill_value=vocabs["mode_size"],     dtype=torch.long, device=device) # BOS

    acts, modes = [], []
    model.eval()
    with torch.no_grad():
        for t in range(96):
            la, lm = model(prof_tokens, x_act, x_mode)
            # 采样（可用 top-k/top-p；这里用贪心演示）
            next_act  = la[:,-1].argmax(-1, keepdim=True)
            next_mode = lm[:,-1].argmax(-1, keepdim=True)

            # 简单规则（夜间更偏 home/stay，可自行增强为掩码）
            if t >= 88 or t <= 8:
                # 强化在家与静止
                next_act  = torch.clamp(next_act, max=cat_maps["activity"]["home"])
                next_mode = torch.clamp(next_mode, max=cat_maps["mode"]["stay"])

            acts.append(int(next_act.item()))
            modes.append(int(next_mode.item()))

            x_act  = torch.cat([x_act,  next_act],  dim=1)
            x_mode = torch.cat([x_mode, next_mode], dim=1)

    return acts, modes

# 生成 100 个“虚拟人”的一天
rows = []
subset = profiles_syn.sample(100, random_state=0).reset_index(drop=True)
for i, p in subset.iterrows():
    act_ids, mode_ids = generate_day_for_profile(p)
    for t in range(96):
        rows.append({
            "pid": int(p.get("pid", i)),  # 如果没有 pid 字段，就用 i
            "t": t,
            "act_id": act_ids[t],
            "mode_id": mode_ids[t]
        })
gen_bins = pd.DataFrame(rows)

# 反映射成可读文本
inv_activity = {v:k for k,v in cat_maps["activity"].items()}
inv_mode     = {v:k for k,v in cat_maps["mode"].items()}
gen_bins["activity"] = gen_bins["act_id"].map(inv_activity)
gen_bins["mode"]     = gen_bins["mode_id"].map(inv_mode)

gen_bins.to_csv("bins_generated_from_profiles_syn.csv", index=False)
subset.to_csv("profiles_generated_base.csv", index=False)  # 采样出来用于生成的“虚拟人”清单
gen_bins.head()


# 真实 vs 生成：对比评估与可视化

我们从四类维度比较：
① 边际与条件分布（出行方式份额，按小时/按年龄组）；
② 序列特征（首发时间、回家时间的分布）；
③ 出行次数/活动切换次数；
④ 简单距离隐私（最近邻相似度）——示意。

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

bins_real = pd.read_csv("bins_real.csv")
gen_bins  = pd.read_csv("bins_generated_from_profiles_syn.csv")

def share_by_hour(df, col, all_categories):
    out = []
    for h in range(24):
        # 该小时对应的 bins：h*4 .. h*4+3
        idx = (df["t"]>=h*4) & (df["t"]<(h+1)*4)
        part = df.loc[idx]
        share = part[col].value_counts(normalize=True)
        # 统一使用所有类别进行重新索引，确保维度一致
        share = share.reindex(all_categories, fill_value=0.0)
        out.append(share.values)
    return np.vstack(out)

# 获取真实数据中所有唯一的出行模式，作为统一的参考类别列表
modes_for_plotting = sorted(bins_real["mode"].unique())

# ① 出行方式份额（按小时）
real_mode_hour = share_by_hour(bins_real, "mode", modes_for_plotting)
gen_mode_hour  = share_by_hour(gen_bins,  "mode", modes_for_plotting)

plt.figure(figsize=(9,5))
for i, m in enumerate(modes_for_plotting):
    plt.plot(real_mode_hour[:,i], label=f"Real-{m}", linestyle="--")
    plt.plot(gen_mode_hour[:,i],  label=f"Gen-{m}")
plt.title("Mode share by hour (Real vs Generated)")
plt.xlabel("Hour"); plt.ylabel("Share"); plt.legend(); plt.grid(True)
plt.show()

# ② 首次离家时间 / 回家时间（简化：home -> 非home 的首次切换时刻；反之亦然）
def first_depart_time(df):
    # 近似：找到 t 从 activity=home 到 !=home 的第一个 t
    dep = df.pivot_table(index="pid", columns="t", values="activity", aggfunc="first")
    firsts = []
    for pid, row in dep.iterrows():
        arr = row.values
        t0 = None
        for t in range(1, len(arr)):
            if arr[t-1]=="home" and arr[t]!="home":
                t0 = t; break
        if t0 is not None: firsts.append(t0/4.0)  # 转为小时
    return np.array(firsts)

def last_arrival_home(df):
    arrv = df.pivot_table(index="pid", columns="t", values="activity", aggfunc="first")
    lasts = []
    for pid, row in arrv.iterrows():
        arr = row.values
        t0 = None
        for t in range(1, len(arr)):
            if arr[t-1]!="home" and arr[t]=="home":
                t0 = t
        if t0 is not None: lasts.append(t0/4.0)
    return np.array(lasts)

r_dep = first_depart_time(bins_real)
g_dep = first_depart_time(gen_bins)
r_arr = last_arrival_home(bins_real)
g_arr = last_arrival_home(gen_bins)

plt.figure(figsize=(9,4))
plt.hist(r_dep, bins=24, alpha=0.5, label="Real depart")
plt.hist(g_dep, bins=24, alpha=0.5, label="Gen depart")
plt.title("First departure time (hour)"); plt.legend(); plt.show()

plt.figure(figsize=(9,4))
plt.hist(r_arr, bins=24, alpha=0.5, label="Real return")
plt.hist(g_arr, bins=24, alpha=0.5, label="Gen return")
plt.title("Last arrival home time (hour)"); plt.legend(); plt.show()

# ③ 每日活动切换次数（近似衡量“出行/活动片段”数量）
def switches_per_day(df):
    cnt = df.sort_values(["pid","t"]).groupby("pid")["activity"].apply(
        lambda x: (x!=x.shift(1)).sum()
    )
    return cnt.values

r_sw = switches_per_day(bins_real)
g_sw = switches_per_day(gen_bins)

plt.figure(figsize=(9,4))
plt.hist(r_sw, bins=30, alpha=0.5, label="Real")
plt.hist(g_sw, bins=30, alpha=0.5, label="Gen")
plt.title("Activity switches per day"); plt.legend(); plt.show()

# ④ 简单“相似度”检查（隐私粗检）：将生成样本在 (hour, mode share vector) 空间与真实分布对比
def hour_mode_vector(df, modes):
    out = []
    for pid, grp in df.groupby("pid"):
        v = []
        for h in range(24):
            idx = (grp["t"]>=h*4) & (grp["t"]<(h+1)*4)
            share = grp.loc[idx, "mode"].value_counts(normalize=True).reindex(modes, fill_value=0.0)
            v.extend(share.values.tolist())
        out.append(np.array(v))
    return np.vstack(out)

modes = sorted(bins_real["mode"].unique().tolist())
R = hour_mode_vector(bins_real, modes)
G = hour_mode_vector(gen_bins,  modes)

# 最近邻距离（生成 → 真实）
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(R)
dist, _ = nn.kneighbors(G)
plt.figure(figsize=(7,4))
plt.hist(dist, bins=40)
plt.title("Nearest-neighbor distance of Generated to Real (lower could signal overfitting)")
plt.show()